# Porto taxi trajectory rasterization with a custom merge

This notebook rasterizes 200,000 taxi trajectories from Porto, Portugal as
GPS-traced lines. Unlike simple pickup-to-dropoff lines, these are full
trajectories with a GPS reading every 15 seconds, so they follow actual streets.

We compare the built-in `sum` merge against a custom log-sum merge that
compresses dynamic range, letting minor streets stand out alongside the main
corridors.

Data source: [ECML/PKDD 2015 Taxi Trajectory Challenge (UCI ML Repository)](https://archive.ics.uci.edu/dataset/339/)

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString

import xrspatial  # registers .xrs accessor
from xrspatial.utils import ngjit

## Download and parse trajectories

The dataset is from the UCI ML Repository (~509 MB download, nested ZIP).
Each trip has a `POLYLINE` column: a JSON list of `[longitude, latitude]` pairs
recorded every 15 seconds. We read the first 200,000 trips to keep things
manageable.

In [ ]:
import urllib.request, zipfile, io, tempfile, os

url = ('https://archive.ics.uci.edu/static/public/339/'
       'taxi+service+trajectory+prediction+challenge+ecml+pkdd+2015.zip')

print('Downloading Porto taxi data (~509 MB)...')
tmpfile, _ = urllib.request.urlretrieve(url)

# Nested ZIP: outer contains train.csv.zip, which contains train.csv
tmpdir = tempfile.mkdtemp()
with zipfile.ZipFile(tmpfile) as outer:
    outer.extract('train.csv.zip', tmpdir)

with zipfile.ZipFile(os.path.join(tmpdir, 'train.csv.zip')) as inner:
    with inner.open('train.csv') as f:
        df = pd.read_csv(f, nrows=200_000,
                         usecols=['POLYLINE', 'MISSING_DATA'])

os.remove(tmpfile)
print(f'{len(df):,} rows read')

In [ ]:
# Drop trips with missing GPS data
df = df[df.MISSING_DATA == False].copy()

# Parse polyline JSON, keep trips with at least 2 GPS points
df['coords'] = df.POLYLINE.apply(json.loads)
df = df[df.coords.apply(len) >= 2].copy()

# Trip duration in minutes (15 seconds between readings)
df['duration_min'] = df.coords.apply(lambda c: (len(c) - 1) * 0.25)

# Build LineStrings
df['geometry'] = df.coords.apply(LineString)
gdf = gpd.GeoDataFrame(df[['duration_min', 'geometry']],
                        geometry='geometry', crs='EPSG:4326')

print(f'{len(gdf):,} trajectories after filtering')
print(f'Duration: min={gdf.duration_min.min():.1f}, '
      f'median={gdf.duration_min.median():.1f}, '
      f'max={gdf.duration_min.max():.1f} minutes')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_facecolor('#1a1a2e')
gdf.sample(min(20_000, len(gdf)), random_state=42).plot(
    ax=ax, linewidth=0.1, alpha=0.3, color='#e94560')
ax.set_title(f'{len(gdf):,} taxi trajectories in Porto')
ax.set_axis_off()
plt.tight_layout()

In [ ]:
# Porto bounding box
bounds = (-8.72, 41.10, -8.52, 41.22)
width, height = 800, 600

def make_template(w, h, bounds):
    xmin, ymin, xmax, ymax = bounds
    px, py = (xmax - xmin) / w, (ymax - ymin) / h
    x = np.linspace(xmin + px / 2, xmax - px / 2, w)
    y = np.linspace(ymax - py / 2, ymin + py / 2, h)
    return xr.DataArray(np.zeros((h, w)), dims=['y', 'x'],
                        coords={'y': y, 'x': x})

template = make_template(width, height, bounds)

## Rasterize: built-in sum vs. custom log-sum

We burn the trajectories into the grid three ways:

1. **Trip count** (`merge='count'`): how many trajectories cross each pixel
2. **Total duration** (`merge='sum'` on `duration_min`): taxi-minutes per pixel
3. **Log-duration** (custom merge): `sum(log(1 + duration))` per pixel

The log transform compresses the contribution of long trips. A 60-minute
airport run contributes `log(61) ~ 4.1` while a 5-minute hop contributes
`log(6) ~ 1.8` -- a 2:1 ratio instead of 12:1. This lets neighborhood streets
where short trips happen show up alongside the highway corridors.

## Results

In [ ]:
# 1. Trip count
count_raster = template.xrs.rasterize(gdf, merge='count', fill=0)

# 2. Total duration (linear sum)
dur_raster = template.xrs.rasterize(gdf, column='duration_min', merge='sum', fill=0)

# 3. Log-duration (custom non-linear merge)
@ngjit
def log_duration_sum(pixel, props, is_first):
    """Sum log(1 + duration) instead of raw duration.

    Compresses the contribution of long trips: a 60-min airport run
    contributes log(61) ~ 4.1 while a 5-min hop contributes log(6) ~ 1.8.
    Ratio drops from 12:1 to about 2:1.
    """
    val = np.log1p(props[0])
    if is_first:
        return val
    return pixel + val

log_raster = template.xrs.rasterize(
    gdf, column='duration_min', merge=log_duration_sum, fill=0)

In [ ]:
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(20, 8), facecolor='black')

titles = ['Trip count', 'Total duration (sum)', 'Log-duration (custom merge)']
rasters = [count_raster, dur_raster, log_raster]

for ax, raster, title in zip(axes, rasters, titles):
    ax.set_facecolor('black')
    masked = raster.where(raster > 0)
    masked.plot.imshow(ax=ax, cmap='hot', add_colorbar=False,
                       interpolation='nearest')
    ax.set_title(title, color='white', fontsize=14, pad=10)
    ax.set_axis_off()

plt.tight_layout()
plt.savefig('images/porto_taxi_lines_preview.png',
            bbox_inches='tight', dpi=120, facecolor='black')

The left and center panels are dominated by the main highway corridors and the
downtown waterfront area. Side streets and residential neighborhoods barely
register.

The right panel applies `log(1 + x)` before summing, compressing the
thousand-fold difference between highways and side streets down to about 3--4x.
You can trace individual neighborhood roads and see the full street network.
Same data, different merge function.

**When to use a non-linear merge:** any time a few features dominate the signal
and you want to see the rest of the distribution. Log-sum works well for data
that spans several orders of magnitude.

### References

- [ECML/PKDD 2015 Taxi Trajectory Challenge (UCI)](https://archive.ics.uci.edu/dataset/339/)
- [Bresenham's line algorithm (Wikipedia)](https://en.wikipedia.org/wiki/Bresenham%27s_line_algorithm)
- [xrspatial.rasterize API docs](https://xarray-spatial.readthedocs.io/en/latest/reference/_autosummary/xrspatial.rasterize.html)